In [2]:
import os
import numpy as np
import pandas as pd

from tqdm import tqdm
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

# ============================================================
# Paths
# ============================================================
project_dir = r"C:\Users\Anna_Maksymchuk1\Desktop\GNN_ESM_Core"

refined_dir = os.path.join(project_dir, "refined_set")
core_index_file = os.path.join(
    project_dir,
    "PDBbind_2016",
    "index",
    "INDEX_core_data.2016",
)

# ============================================================
# Helpers
# ============================================================
def get_pdb_ids_from_index(filename):
    pdb_ids = set()

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            pdb_ids.add(line.split()[0].lower())

    return pdb_ids


def load_ligand(refined_dir, pdb_id):
    mol2_path = os.path.join(
        refined_dir,
        pdb_id,
        f"{pdb_id}_ligand.mol2",
    )

    if os.path.exists(mol2_path):
        mol = Chem.MolFromMol2File(
            mol2_path,
            sanitize=True,
            removeHs=False,
        )
        if mol is not None:
            return mol

    sdf_path = os.path.join(
        refined_dir,
        pdb_id,
        f"{pdb_id}_ligand.sdf",
    )

    if os.path.exists(sdf_path):
        supplier = Chem.SDMolSupplier(
            sdf_path,
            sanitize=True,
            removeHs=False,
        )
        if supplier is not None and len(supplier) > 0:
            return supplier[0]

    return None


def mol_to_fingerprint(mol, radius=2, n_bits=2048):
    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=radius,
        nBits=n_bits,
    )


# ============================================================
# Build train/core ID sets
# ============================================================
core_ids = get_pdb_ids_from_index(core_index_file)

all_refined_ids = {
    folder.lower()
    for folder in os.listdir(refined_dir)
    if os.path.isdir(os.path.join(refined_dir, folder))
    and not folder.startswith(".")
}

# 1pxn may be absent after preprocessing, but here we inspect files on disk
core_ids_present = sorted(core_ids & all_refined_ids)
train_ids = sorted(all_refined_ids - core_ids)

print(f"Refined folders:       {len(all_refined_ids)}")
print(f"Core IDs in index:     {len(core_ids)}")
print(f"Core IDs present:      {len(core_ids_present)}")
print(f"Refined minus core:    {len(train_ids)}")

# ============================================================
# Load fingerprints
# ============================================================
train_fps = {}
failed_train = []

for pdb_id in tqdm(train_ids, desc="Loading train ligands"):
    mol = load_ligand(refined_dir, pdb_id)

    if mol is None:
        failed_train.append(pdb_id)
        continue

    train_fps[pdb_id] = mol_to_fingerprint(mol)

core_fps = {}
failed_core = []

for pdb_id in tqdm(core_ids_present, desc="Loading core ligands"):
    mol = load_ligand(refined_dir, pdb_id)

    if mol is None:
        failed_core.append(pdb_id)
        continue

    core_fps[pdb_id] = mol_to_fingerprint(mol)

print(f"Train ligands loaded: {len(train_fps)}")
print(f"Core ligands loaded:  {len(core_fps)}")
print(f"Failed train ligands: {failed_train}")
print(f"Failed core ligands:  {failed_core}")

# ============================================================
# For each core ligand, find nearest train ligand
# ============================================================
train_pdb_ids = list(train_fps.keys())
train_fp_list = [train_fps[pdb_id] for pdb_id in train_pdb_ids]

rows = []

for core_pdb_id, core_fp in tqdm(core_fps.items(), desc="Computing ligand similarity"):
    sims = DataStructs.BulkTanimotoSimilarity(core_fp, train_fp_list)

    best_idx = int(np.argmax(sims))
    max_sim = float(sims[best_idx])
    nearest_train_pdb_id = train_pdb_ids[best_idx]

    rows.append(
        {
            "core_pdb_id": core_pdb_id,
            "nearest_train_pdb_id": nearest_train_pdb_id,
            "max_ligand_tanimoto": max_sim,
        }
    )

ligand_similarity_df = pd.DataFrame(rows).sort_values(
    "max_ligand_tanimoto",
    ascending=False,
)

# ============================================================
# Summary
# ============================================================
print("\nLigand similarity summary:")
print(ligand_similarity_df["max_ligand_tanimoto"].describe())

print("\nCounts above thresholds:")
for threshold in [0.7, 0.8, 0.9, 0.95]:
    count = (ligand_similarity_df["max_ligand_tanimoto"] >= threshold).sum()
    print(f"Tanimoto >= {threshold:.2f}: {count}")

display(ligand_similarity_df.head(30))

# Optional save
out_path = os.path.join(project_dir, "outputs", "core_vs_refined_minus_core_ligand_similarity.csv")
ligand_similarity_df.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")

Refined folders:       4056
Core IDs in index:     290
Core IDs present:      290
Refined minus core:    3766


Loading train ligands:   0%|          | 0/3766 [00:00<?, ?it/s][23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGenerator
[23:42:23] DEPRECATION WARNING: please use MorganGen

Train ligands loaded: 3763
Core ligands loaded:  290
Failed train ligands: ['1ksn', '1wc1', '5tmp']
Failed core ligands:  []


Computing ligand similarity: 100%|██████████| 290/290 [00:00<00:00, 324.93it/s]


Ligand similarity summary:
count    290.000000
mean       0.642038
std        0.243885
min        0.208589
25%        0.430804
50%        0.632704
75%        0.831790
max        1.000000
Name: max_ligand_tanimoto, dtype: float64

Counts above thresholds:
Tanimoto >= 0.70: 112
Tanimoto >= 0.80: 79
Tanimoto >= 0.90: 64
Tanimoto >= 0.95: 61


,core_pdb_id,nearest_train_pdb_id,max_ligand_tanimoto
250,4jsz,2wej,1.0
242,4ivb,4iva,1.0
233,4f3c,3dp9,1.0
232,4f2w,1y6q,1.0
231,4f09,4ehz,1.0
48,2cbv,4iie,1.0
235,4gfm,4gfo,1.0
230,4eor,1h1s,1.0
201,3wtj,3c84,1.0
194,3uex,1hmr,1.0



Saved to: C:\Users\Anna_Maksymchuk1\Desktop\GNN_ESM_Core\outputs\core_vs_refined_minus_core_ligand_similarity.csv


In [3]:
high_sim = ligand_similarity_df[
    ligand_similarity_df["max_ligand_tanimoto"] >= 0.95
].sort_values("max_ligand_tanimoto", ascending=False)

print(high_sim.to_string(index=False))

core_pdb_id nearest_train_pdb_id  max_ligand_tanimoto
       4jsz                 2wej             1.000000
       4ivb                 4iva             1.000000
       4f3c                 3dp9             1.000000
       4f2w                 1y6q             1.000000
       4f09                 4ehz             1.000000
       2cbv                 4iie             1.000000
       4gfm                 4gfo             1.000000
       4eor                 1h1s             1.000000
       3wtj                 3c84             1.000000
       3uex                 1hmr             1.000000
       3uev                 1hmr             1.000000
       3uew                 1hmr             1.000000
       1w4o                 1n3z             1.000000
       1y6r                 1k27             1.000000
       4dld                 3tza             1.000000
       1uto                 1utm             1.000000
       1syi                 1syh             1.000000
       2j7h                 

In [5]:
exact_like = ligand_similarity_df[
    ligand_similarity_df["max_ligand_tanimoto"] == 1.0
]

print(f"Pairs with Tanimoto = 1.0: {len(exact_like)}")
print(exact_like.to_string(index=False))

Pairs with Tanimoto = 1.0: 59
core_pdb_id nearest_train_pdb_id  max_ligand_tanimoto
       4jsz                 2wej                  1.0
       4ivb                 4iva                  1.0
       4f3c                 3dp9                  1.0
       4f2w                 1y6q                  1.0
       4f09                 4ehz                  1.0
       2cbv                 4iie                  1.0
       4gfm                 4gfo                  1.0
       4eor                 1h1s                  1.0
       3wtj                 3c84                  1.0
       3uex                 1hmr                  1.0
       3uev                 1hmr                  1.0
       3uew                 1hmr                  1.0
       1w4o                 1n3z                  1.0
       1y6r                 1k27                  1.0
       4dld                 3tza                  1.0
       1uto                 1utm                  1.0
       1syi                 1syh                  1.

In [6]:
import os
import pandas as pd

project_dir = r"C:\Users\Anna_Maksymchuk1\Desktop\GNN_ESM_Core"

seq_path = os.path.join(project_dir, "protein_sequences.csv")
df_seq = pd.read_csv(seq_path)

df_seq["pdb_id"] = df_seq["pdb_id"].str.lower()

core_ids = get_pdb_ids_from_index(
    os.path.join(
        project_dir,
        "PDBbind_2016",
        "index",
        "INDEX_core_data.2016",
    )
)

core_seq_df = df_seq[df_seq["pdb_id"].isin(core_ids)].copy()
train_seq_df = df_seq[~df_seq["pdb_id"].isin(core_ids)].copy()

# Если у тебя несколько цепей разделены ':',
# пока можно сравнивать полный sequence string как есть.
train_seq_to_ids = (
    train_seq_df.groupby("sequence")["pdb_id"]
    .apply(list)
    .to_dict()
)

rows = []

for _, row in core_seq_df.iterrows():
    seq = row["sequence"]
    matching_train_ids = train_seq_to_ids.get(seq, [])

    rows.append(
        {
            "core_pdb_id": row["pdb_id"],
            "has_exact_sequence_match_in_train": len(matching_train_ids) > 0,
            "matching_train_ids": matching_train_ids,
        }
    )

exact_seq_df = pd.DataFrame(rows)

print("Exact protein sequence matches in train:")
print(exact_seq_df["has_exact_sequence_match_in_train"].value_counts())

display(
    exact_seq_df[
        exact_seq_df["has_exact_sequence_match_in_train"]
    ].head(30)
)

Exact protein sequence matches in train:
has_exact_sequence_match_in_train
False    149
True     138
Name: count, dtype: int64


,core_pdb_id,has_exact_sequence_match_in_train,matching_train_ids
0,1a30,True,"[1d4y, 1hpo, 1mrw, 1msm, 2pk5, 2pk6, 3kdb, 3kd..."
1,1bcu,True,"[1qbv, 1t4v, 2r2m]"
3,1c5z,True,[1gi7]
4,1eby,True,"[1hxw, 1izh, 1pro, 1sbg, 4ll3]"
9,1k1i,True,"[1bju, 1bjv, 1bty, 1c1r, 1c5p, 1c5q, 1c5s, 1c5..."
10,1lpg,True,"[1lpk, 1lpz]"
11,1mq6,True,[1mq5]
12,1nc1,True,[1y6q]
14,1nvq,True,"[1nvr, 1nvs]"
15,1o0h,True,"[1afk, 1afl, 1jn4, 1jvu, 1o0f, 1o0m, 1o0n, 1qh..."
